# Atlas — research examplesA guided tour of the Atlas API. Every cell runs offline using the deterministic**synthetic** provider; switch `PROVIDER` to `"yfinance"` for real history.> **Synthetic data is simulated, not observed.** Results below demonstrate the> machinery and say nothing about real market performance.

In [ ]:
import sys, datetime as dtfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent / "src"))import numpy as npimport pandas as pdfrom atlas.config import load_configfrom atlas.logging_utils import configure_loggingconfigure_logging("WARNING", force=True)PROVIDER = "synthetic"          # "yfinance" for real dataconfig = load_config(Path.cwd().parent / "configs")config.assets.data.provider = PROVIDERprint("universe:", config.symbols)print("config hash:", config.config_hash)

## 1. Load and validate market data

In [ ]:
from atlas.data.loader import MarketDataLoaderfrom atlas.data.validator import DataValidatorloader = MarketDataLoader.from_config(config)panel = loader.load(config.universe.symbols, dt.date(2008, 1, 1), dt.date(2018, 12, 31))panel.describe()

In [ ]:
report = DataValidator(min_history_days=260).validate(panel, expected_symbols=config.symbols)print(report.render(max_rows=15))

## 2. FeaturesEvery feature is backward-looking. `tests/unit/test_lookahead.py` proves it byperturbing future prices and asserting nothing historical moves.

In [ ]:
from atlas.data.features import FeatureEngineerfeatures = FeatureEngineer.from_config(config).build(panel)print(features.summary())sorted(features.names)[:15]

In [ ]:
# Demonstrate the point-in-time property directly.cut = panel.dates[1500]full = features.get("return_252d").loc[:cut]realtime = FeatureEngineer.from_config(config).build(panel.as_of(cut)).get("return_252d")common = full.index.intersection(realtime.index)print("identical on", len(common), "dates:",      np.allclose(full.loc[common].to_numpy(), realtime.loc[common].to_numpy(), equal_nan=True))

## 3. Strategy signals

In [ ]:
from atlas.strategies.base import build_strategiesstrategies = build_strategies(config)signals = {s.name: s.compute(panel.adj_close, features) for s in strategies}pd.DataFrame([r.summary() for r in signals.values()]).set_index("strategy").round(3)

In [ ]:
# The tidy record form required by the Strategy interface.strategies[0].generate_signals(panel.adj_close, features).head()

## 4. Regime detection

In [ ]:
from atlas.regimes.rules_based import RulesBasedRegimeDetectorregimes = RulesBasedRegimeDetector(config.regimes).detect(panel, features)print(regimes.distribution().round(3))regimes.episodes().sort_values("days", ascending=False).head()

## 5. Combine signals

In [ ]:
from atlas.strategies.ensemble import SignalEnsembleensemble = SignalEnsemble(strategies, config.strategies.ensemble, regime_config=config.regimes)combined = ensemble.combine(signals, regimes=regimes)combined.contribution_share().mean().round(3)

## 6. Portfolio construction

In [ ]:
from atlas.portfolio.allocator import PortfolioAllocatorallocator = PortfolioAllocator(config)as_of = panel.dates[-1]risk_multiplier, defensive = allocator.risk_posture(regimes.label_at(as_of))allocation = allocator.allocate(    combined.signal.loc[as_of],    panel.returns().loc[:as_of].tail(520),    risk_multiplier=risk_multiplier,    defensive_weight=defensive,)print(allocation.summary())allocation.weights[allocation.weights.abs() > 1e-6].sort_values(ascending=False).round(4)

In [ ]:
# Every constraint adjustment carries a reason.allocation.constraints.to_frame()

## 7. Backtest

In [ ]:
from atlas.backtest.engine import BacktestEngineresult = BacktestEngine(config).run(panel, features=features)print(result.metrics.render())

In [ ]:
result.cost_breakdown().round(2)

In [ ]:
pd.Series(result.cost_drag()).round(4)

## 8. Benchmarks

In [ ]:
from atlas.validation.benchmarks import BenchmarkSuitebenchmarks = BenchmarkSuite(config).run(panel, atlas_result=result, features=features)benchmarks.table().round(3)

## 9. Charts

In [ ]:
from atlas.reporting import chartscurves = {"Atlas": result.equity_curve}for name, series in benchmarks.returns.items():    if name != "Atlas regime-aware ensemble":        curves[name] = (1 + series).cumprod()charts.equity_curve_chart(curves)

In [ ]:
charts.drawdown_chart(result.equity_curve)

In [ ]:
charts.regime_timeline_chart(regimes.labels, equity=result.equity_curve)

## 10. Walk-forward validationThe only genuinely out-of-sample result. Slow: one backtest per candidate perwindow.

In [ ]:
from atlas.validation.walk_forward import WalkForwardValidatorwf_config = config.model_copy(deep=True)wf_config.validation.walk_forward.train_years = 3wf_config.validation.walk_forward.step_years = 2wf_config.validation.walk_forward.parameter_grid = {"trend.slow_ma": [150, 200]}walk_forward = WalkForwardValidator(wf_config).run(panel, features=features, progress=False)print(walk_forward.test_metrics.render() if walk_forward.test_metrics else "no OOS returns")

In [ ]:
# The number that matters most: how much performance degrades out of sample.pd.Series(walk_forward.degradation()).round(3)

## 11. Stress periods and bootstrap

In [ ]:
from atlas.validation.stress_tests import StressTester, block_bootstrap, bootstrap_statisticsstress = StressTester(config.validation).run(result, panel=panel)stress.periods[["name", "total_return", "max_drawdown", "worst_day", "regime_mix"]].round(4)

In [ ]:
paths = block_bootstrap(result.returns, n_simulations=300, block_size=21, horizon=252, seed=7)bootstrap_statistics(paths, [0.05, 0.25, 0.5, 0.75, 0.95]).round(4)

## 12. Dry-run order preview

In [ ]:
from atlas.execution.ibkr_client import MockBrokerClientfrom atlas.execution.order_manager import OrderManagerbroker = MockBrokerClient(prices=panel.adj_close.iloc[-1], initial_cash=config.portfolio.initial_capital)broker.connect()cycle = OrderManager(config, broker=broker).run_cycle(panel, submit=False)print(cycle.plan.render())broker.disconnect()

---**Disclaimer.** Atlas is an educational research and paper-trading system.Historical performance does not guarantee future results. Paper-tradingexecution does not replicate all real-world liquidity, slippage, latency, andfill conditions.